# 08 — Retrieval-Augmented Reliability

RAG-based systems can be evaluated for citation accuracy, grounding, and faithfulness to retrieved context.

## Citation Accuracy, Grounding Scores, RAG Faithfulness

RAG systems have two distinct failure modes:
1. **Retrieval failures**: wrong documents are retrieved → no amount of faithful generation helps
2. **Faithfulness failures**: correct documents are retrieved but the LLM ignores or contradicts them

**Faithfulness metrics**:
- *Faithfulness score*: for each claim in the response, check if it's supported by the cited documents
- *Attribution accuracy*: does each cited source actually support the sentence citing it?
- *Hallucination rate*: fraction of response claims not grounded in any retrieved document

**RAGAS** (Es et al., 2023) is a RAG evaluation framework measuring:
- *Faithfulness*: claims in the answer / claims in the answer supported by context
- *Answer Relevance*: how relevant the answer is to the question
- *Context Recall*: how much of the ground truth is in the retrieved context
- *Context Precision*: fraction of retrieved context that is relevant

**Citation accuracy** for attributed generation (e.g., 'According to [1], X is true'): verify that the cited document actually says X — LLMs frequently hallucinate citations.

In [ ]:
# RAG faithfulness evaluator
import numpy as np

# Simulated RAG pipeline
KNOWLEDGE_BASE = [
    {'id': 'doc1', 'text': 'Paris is the capital of France and was founded by the Romans.'},
    {'id': 'doc2', 'text': 'The Eiffel Tower was built for the 1889 World Fair in Paris.'},
    {'id': 'doc3', 'text': 'France has a population of approximately 68 million people.'},
    {'id': 'doc4', 'text': 'The French Revolution began in 1789 and ended the monarchy.'},
    {'id': 'doc5', 'text': 'French cuisine is known for baguettes, croissants, and wine.'},
]

def simple_nli(claim, source_text):
    """Simplified NLI: check if key terms from claim appear in source."""
    claim_words = set(claim.lower().split())
    source_words = set(source_text.lower().split())
    # Remove stop words
    stops = {'the', 'a', 'an', 'is', 'was', 'of', 'in', 'and', 'for', 'by', 'it'}
    claim_key = claim_words - stops
    overlap = len(claim_key & source_words) / max(len(claim_key), 1)
    return overlap > 0.3

def compute_faithfulness(response_claims, retrieved_docs):
    """
    For each claim in the response, check if any retrieved doc supports it.
    Returns faithfulness score (0-1).
    """
    supported = 0
    details = []
    for claim in response_claims:
        supported_by = []
        for doc in retrieved_docs:
            if simple_nli(claim, doc['text']):
                supported_by.append(doc['id'])
        if supported_by:
            supported += 1
        details.append({'claim': claim[:50], 'supported_by': supported_by})
    return supported / max(len(response_claims), 1), details

# Test cases
test_cases = [
    {
        'question': 'What is Paris known for?',
        'retrieved': [KNOWLEDGE_BASE[0], KNOWLEDGE_BASE[1]],
        'response_claims': [
            'Paris is the capital of France',
            'The Eiffel Tower is in Paris built in 1889',
            'Paris has many famous museums',  # not in retrieved docs
        ]
    },
    {
        'question': 'What is France\'s population?',
        'retrieved': [KNOWLEDGE_BASE[2]],
        'response_claims': [
            'France has about 68 million people',
            'France is one of the largest European economies',  # hallucinated
        ]
    },
]

for tc in test_cases:
    score, details = compute_faithfulness(tc['response_claims'], tc['retrieved'])
    print(f'Q: {tc["question"]}')
    print(f'Faithfulness: {score:.2f}')
    for d in details:
        status = 'SUPPORTED' if d['supported_by'] else 'HALLUCINATED'
        print(f'  [{status}] "{d["claim"]}"')
        if d['supported_by']:
            print(f'    -> {d["supported_by"]}')
    print()